# B-Free x GlobalForge — Notebook 01: Setup Bundle Builder (K1)

Chạy trên session **CPU hoặc T4×2, Internet ON**. Notebook này **tạo** các assets offline cho notebooks 02/03 (những notebook chạy trên RTX PRO 6000 Blackwell 96GB, hoàn toàn offline — không pip internet, không HF hub):

1. `/kaggle/working/bfree-wheels/` — wheel bundle cho **Linux x86_64, Python 3.11** (Kaggle dùng chung image Linux x86_64 Python 3.11 cho mọi accelerator, nên wheels tải ở session CPU/T4 khớp session RTX PRO 6000):
   `torch==2.8.0+cu128`, `torchvision==0.23.0+cu128` (Blackwell sm_120 cần >=2.8 — risk table), `timm==1.0.22`, `peft==0.15.2`, `transformers==4.55.4`, `pandas==2.3.3` (MUST <3), `numpy==1.26.4`, `matplotlib==3.11.1`, `seaborn==0.13.2`, `scikit-learn`, `scipy`, `pyyaml`, `pillow`, `tqdm`, `safetensors`, `huggingface_hub` (+ dependencies).
2. `/kaggle/working/dinov2-vitb14-reg4-pretrain/` — weights DINOv2 ViT-B/14 reg4 (`timm/vit_base_patch14_reg4_dinov2.lvd142m`, `model.safetensors`) cho offline pretrained init (D5) của notebook 02.

Notebook 02/03 sẽ `pip install --no-index` từ bundle này (rule agent.md: offline, không apt-get, không HF hub).

Sau khi chạy xong: **Save Version → Save & Run All (Commit)**, rồi từ tab **Output** tạo 2 Kaggle Datasets (`bfree-wheels`, `dinov2-vitb14-reg4-pretrain`) và attach vào notebooks 02/03.

In [ ]:
import platform
import sys

print(f"Python  : {sys.version.split()[0]} (bundle target: Kaggle Linux x86_64, Python 3.11)")
print(f"Platform: {platform.platform()}")
try:
    import torch
    print(f"torch (session, preinstalled): {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} — not required here (CPU session is fine)")
except ImportError:
    print("torch not preinstalled in this session — fine, this notebook only downloads wheels.")

In [ ]:
WHEELS_DIR = "/kaggle/working/bfree-wheels"

!pip download --quiet --dest {WHEELS_DIR} \
    --index-url https://download.pytorch.org/whl/cu128 \
    --extra-index-url https://pypi.org/simple \
    torch==2.8.0+cu128 torchvision==0.23.0+cu128

!pip download --quiet --dest {WHEELS_DIR} \
    timm==1.0.22 peft==0.15.2 transformers==4.55.4 \
    pandas==2.3.3 numpy==1.26.4 matplotlib==3.11.1 seaborn==0.13.2 \
    scikit-learn scipy pyyaml pillow tqdm safetensors huggingface_hub

import glob
import os

wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
assert wheels, "pip download produced no wheels."
print(f"{len(wheels)} wheels, {sum(os.path.getsize(w) for w in wheels) / 1024**3:.2f} GB -> {WHEELS_DIR}")

In [ ]:
!pip install --quiet timm==1.0.22 peft==0.15.2 transformers==4.55.4 pyyaml safetensors huggingface_hub

In [ ]:
REPO_URL = "https://github.com/P-Bao/B-Free.git"
REPO_DIR = "/kaggle/working/B-Free"
BRANCH = "integration/loss-backbone"

import os, sys, glob
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already cloned.")

assert os.path.isfile(os.path.join(REPO_DIR, "code", "networks", "bfree_globalforge_vit.py")), "Clone failed: backbone file missing."
stubs = glob.glob(os.path.join(REPO_DIR, "code", "modules", "*_stub.py"))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"
print("OK: repo cloned on integration/loss-backbone, no stub files (K0 verified).")

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_DIR, "code"))

import torch
import timm
import peft
import transformers
import yaml

print(f"torch (session) = {torch.__version__}")
print(f"timm            = {timm.__version__}")
print(f"peft            = {peft.__version__}")
print(f"transformers    = {transformers.__version__}")

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from configs.loader import load_config, build_model_kwargs
from datasets.bfree_dataset import BFreeDataset, DegradationPipeline
from modules.lib_adapter import LIBAdapter
from modules.gsr_adapter import GSRAdapter
from modules.dcs_loss import DCSLoss, info_nce_loss

cfg = load_config(os.path.join(REPO_DIR, "code", "configs", "bfree_dcs.yaml"))
kwargs = build_model_kwargs(cfg)
print("\nModel kwargs from bfree_dcs.yaml:")
for k, v in kwargs.items():
    print(f"  {k} = {v}")
print("\nIMPORT VALIDATION OK")

In [ ]:
import os
import shutil

from huggingface_hub import hf_hub_download

PRETRAIN_DIR = "/kaggle/working/dinov2-vitb14-reg4-pretrain"
os.makedirs(PRETRAIN_DIR, exist_ok=True)

src = hf_hub_download(repo_id="timm/vit_base_patch14_reg4_dinov2.lvd142m",
                      filename="model.safetensors")
dst = os.path.join(PRETRAIN_DIR, "model.safetensors")
shutil.copyfile(src, dst)
print(f"saved: {dst} ({os.path.getsize(dst) / 1024**2:.1f} MB)")

from safetensors.torch import load_file

sd = load_file(dst)
embed_dim = sd["patch_embed.proj.weight"].shape[0]
pe = sd["pos_embed"]
print(f"state dict: {len(sd)} tensors | embed_dim={embed_dim} | pos_embed={tuple(pe.shape)}")
assert embed_dim == 768, "not a ViT-B checkpoint"
assert tuple(pe.shape) == (1, 37 * 37 + 5, 768), (
    f"unexpected pos_embed shape {tuple(pe.shape)} — expected 518px grid (37x37) + 5 prefix tokens")
print("DINOv2 ViT-B/14 reg4 weights OK (518px grid 37x37 + 5 prefix tokens)")

In [ ]:
import torch

model_smoke = BFreeGlobalForgeViT(img_size=224, pretrained=False)
x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model_smoke(x)
print(f"CPU smoke test: logits {tuple(out['logits'].shape)}, cls {tuple(out['cls'].shape)}")
assert out["logits"].shape == (2, 2) and out["cls"].shape == (2, 768)
del model_smoke, x, out
print("MODEL SMOKE OK")

In [ ]:
import glob
import os

PIN_STACK = {
    "torch": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "timm": "1.0.22",
    "peft": "0.15.2",
    "transformers": "4.55.4",
    "pandas": "2.3.3",
    "numpy": "1.26.4",
    "matplotlib": "3.11.1",
    "seaborn": "0.13.2",
}

wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
names = {os.path.basename(w).split("-")[0].replace("_", "-").lower() for w in wheels}
missing = [f"{pkg}=={ver}" for pkg, ver in PIN_STACK.items() if pkg not in names]
assert not missing, f"Missing pinned wheels in bundle: {missing}"
assert os.path.isfile(os.path.join(PRETRAIN_DIR, "model.safetensors")), "model.safetensors missing"

total_gb = sum(os.path.getsize(w) for w in wheels) / 1024**3
print(f"bundle check OK: {len(wheels)} wheels ({total_gb:.2f} GB) + DINOv2 ViT-B/14 reg4 weights")
for w in wheels[:40]:
    print(f"  {os.path.basename(w)}")

## Bundle Ready (K1 pass)

Output trong `/kaggle/working/`:
- `bfree-wheels/` — wheel bundle pin stack đầy đủ (Linux x86_64, Python 3.11) cho `pip install --no-index`
- `dinov2-vitb14-reg4-pretrain/model.safetensors` — DINOv2 ViT-B/14 reg4 pretrained (D5); notebook 02 sẽ resample `pos_embed` 518px→504px khi load

**Bước tiếp theo (bắt buộc):**
1. **Save Version → Save & Run All (Commit)** để chốt output.
2. Tab **Output** của notebook này → **New Dataset** cho từng thư mục:
   - `bfree-wheels/` → dataset `bfree-wheels`
   - `dinov2-vitb14-reg4-pretrain/` → dataset `dinov2-vitb14-reg4-pretrain`
3. Attach 2 dataset đó vào `02_bfree_kaggle_train.ipynb` và `03_bfree_kaggle_eval.ipynb` (chạy offline trên RTX PRO 6000).